# requirements

In [1]:
import numpy as np
import pandas as pd
import sys

# 01.27.2026 more debugging

In [3]:
# debug model loading
import torch
checkpoint_path = "/data/users2/ppopov1/multimodal-subnetworks/logs/old_logs/masked_fbirn_('falff',)_('gender_encoded',)/fold_1/model.best.pth"

from resnet import ResNet3D
from src.masked_model import MultiMaskSNIPWrapper

model_channels = 64 # from config
n_classes = 1 # from config, binary

model = ResNet3D(
    in_channels=1, 
    n_classes=n_classes, 
    channels=model_channels
)
model = MultiMaskSNIPWrapper(
    model,
    sparsity=0.9,
)

model.prepare_for_loading([1])
model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))

Restoring parametrization structure for modalities: [1]


<All keys matched successfully>

# 01.26.2026 deubgging GPU

In [ ]:
# get_data() gives you a test input for the model

import os
from src.db_client import ClientCreator
import numpy as np
import pandas as pd
from mindfultensors.mongoloader import MongoDataset, MongoClient
from mindfultensors.utils import unit_interval_normalize, DBBatchSampler
from catalyst.data import BatchPrefetchLoaderWrapper
from torch.utils.data import DataLoader
import torch

def make_serial(samples_for_id, kind):
    return b"".join(
        [
            sample["chunk"]
            for sample in sorted(
                (
                    sample
                    for sample in samples_for_id
                    if sample["kind"] == kind
                ),
                key=lambda x: x["chunk_id"],
            )
        ]
    )

def multimodal_collate(results, field=("input", "modality", "label")):
    """
    Use this collate function with BatchPrefetchLoaderWrapper when using MultimodalMongoDataset.
    It will stack the inputs, modality codes, and labels into tensors properly.
    """
    modality_mapping = {
        "smri": 0,
        "falff": 1,
        "dwi": 2,
    }

    # Assuming 'results' is your dictionary containing all the data
    input_tensors = [results[id_][field[0]] for id_ in results.keys()]
    modalities = [results[id_][field[1]] for id_ in results.keys()]
    label_tensors = [results[id_][field[2]] for id_ in results.keys()]
    # Stack all input tensors into a single tensor
    stacked_inputs = torch.stack(input_tensors)
    # Stack all label tensors into a single tensor
    stacked_modalities = torch.stack([torch.tensor(modality_mapping[mod]) for mod in modalities])
    stacked_labels = torch.stack(label_tensors)
    return stacked_inputs.unsqueeze(1), stacked_modalities.long(), stacked_labels.float()

def get_data(batch = list(range(1,11))):
    host = "10.245.12.58"
    host_slurm = "arctrdcn018.rs.gsu.edu"
    db_host = host_slurm if os.environ.get("SLURM_JOB_ID") else host
    crop_tensor = False
    client_creator = ClientCreator(
        db_host, crop_tensor=crop_tensor
    )

    db_name = databases = "MindfulTensors"
    db_collection = collections = "fbirn"
    meta_fields = ["gender_encoded"]
    numcubes = 1
    cubesizes = 256
    subvolume_shape = [cubesizes] * 3

    client_creator.set_database(databases)
    client_creator.set_collection(collections)
    client_creator.set_num_subcubes(numcubes)
    client_creator.set_shape(subvolume_shape)

    funcs = {
        "createclient": client_creator.create_client,
        "createVclient": client_creator.create_client,
        "mycollate": client_creator.mycollate,
        "mycollate_full": client_creator.mycollate_full,
        "mytransform": client_creator.mytransform,
    }

    db_fields = ["falff", "smri", "dwi"]

    client = MongoClient("mongodb://" + db_host + ":27017")
    db = client[db_name]
    posts_bin = db[db_collection + ".bin"]
    posts_meta = db[db_collection + ".meta"]



    fields = {"id": 1, "chunk": 1, "kind": 1, "chunk_id": 1}
    samples = list(
        posts_bin.find(
            {
                "id": {"$in": batch},
                "kind": {"$in": db_fields}, # .bin contains 3D kinds like 'smri', 'falff', 'dwi'. Scalar labels are stored in .meta
            },
            fields,
        )
    )

    results = {}

    for id in batch:
        # get ID's label and modalities
        meta_for_id = list(
            posts_meta.find(
                {
                    "id": id,
                },
                meta_fields + ["modalities"],
            )
        )

        assert len(meta_for_id) != 0, f"No meta entries found for id {id}"
        assert len(meta_for_id) < 2, f"More than one meta entry found for id {id}"

        label = meta_for_id[0][meta_fields[0]]
        modalities = meta_for_id[0]["modalities"]
        id_modalities = set(modalities).intersection(set(db_fields))

        # Get samples for this ID
        samples_for_id = [
            sample
            for sample in samples
            if sample["id"] == id
        ]

        for mod in id_modalities:
            data = make_serial(samples_for_id, mod)

            for mod in id_modalities:
                data = make_serial(samples_for_id, mod)

                result = {
                    "input": unit_interval_normalize(funcs["mytransform"](data).float()),
                    "modality": mod,
                    "label": torch.tensor(label).unsqueeze(0),
                }

                results[str(id)+'_'+mod] = result

    return multimodal_collate(results)

In [ ]:
data, modalities, labels = get_data(list(range(1, 20)))

data.shape, modalities.shape, labels.shape

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.parametrize as parametrize
from copy import deepcopy

# Define layers we want to prune
PRUNE_LAYERS = (nn.Linear, nn.Conv3d)

class MultimodalSNIPMask(nn.Module):
    """
    The parametrization module. It holds all masks for a specific layer
    and applies the correct one based on the active_mod_id.
    """
    def __init__(self, masks_dict):
        super().__init__()
        # We register masks as buffers so they move with .to(device) 
        # and don't get gradients.
        self.keys = sorted(list(masks_dict.keys()))
        for mod_id, mask_tensor in masks_dict.items():
            # Name format: mask_{mod_id}
            self.register_buffer(f'mask_{mod_id}', mask_tensor)
        
        # State to control which mask is active. None = Identity (no mask)
        self.active_mod_id = None

    def forward(self, weight):
        if self.active_mod_id is None:
            return weight
        
        # dynamic getattr to find the right buffer
        mask = getattr(self, f'mask_{self.active_mod_id}')
        
        # This multiplication is tracked by autograd. 
        # Backward pass will automatically mask the gradients!
        return weight * mask

class MultiMaskSNIPWrapper(nn.Module):
    def __init__(self, model, sparsity=0.9):
        super(MultiMaskSNIPWrapper, self).__init__()
        
        self.model = model
        self.sparsity = sparsity

        # Helper for SNIP calculation (frozen copy)
        self.cpu_model = deepcopy(model).to('cpu')
        self.cpu_optimizer = torch.optim.SGD(self.cpu_model.parameters(), 0.1)
        self.model_device = next(iter(model.parameters())).device
        
        # Flag to ensure we don't register twice
        self.masks_registered = False

    def register_multimodal_masks(self, modalities, input_data, labels):
        """
        1. Calculates SNIP scores.
        2. Organizes masks by layer.
        3. Registers the parametrization ONCE.
        """
        # 1. Generate Masks Dictionary: {mod_id: {layer_name: mask}}
        temp_mask_storage = {}
        unique_modalities = torch.unique(modalities).cpu().detach().tolist()
        
        for mod in unique_modalities:
            mask_idx = (modalities == mod)
            mod_data = input_data[mask_idx]
            mod_labels = labels[mask_idx]
            
            print(f"Generating masks for modality: {mod}")
            batch = (mod_data, mod_labels)
            masks_by_name, _ = self.generate_mask_from_grad_scores(batch)
            temp_mask_storage[mod] = masks_by_name

        # 2. Register Parametrizations on the actual model
        print("Registering Parametrizations...")
        for name, module in self.model.named_modules():
            if isinstance(module, PRUNE_LAYERS):
                # Collect all masks for THIS specific module
                layer_masks = {}
                has_masks = False
                for mod, mask_dict in temp_mask_storage.items():
                    if name in mask_dict:
                        layer_masks[mod] = mask_dict[name]
                        has_masks = True
                
                if has_masks:
                    # Create the parametrization module
                    snip_mask_module = MultimodalSNIPMask(layer_masks)
                    # Register it! This moves module.weight -> module.parametrizations.weight.original
                    parametrize.register_parametrization(module, "weight", snip_mask_module)
        
        self.masks_registered = True
        print("Done.")

    def forward(self, input_data, modalities):
        if not self.masks_registered:
            print("WARNING: Masks not registered. Using full weights.")
            return self.model(input_data)

        batch_size = input_data.shape[0]
        # Infer output shape or hardcode (e.g. [B, 1] for binary)
        final_outputs = torch.zeros(batch_size, 1, device=self.model_device) 
        
        unique_mods = torch.unique(modalities).cpu().tolist()

        for mod in unique_mods:
            mod_idx = (modalities == mod)
            sub_data = input_data[mod_idx]
            
            # A. Set the Active Modality
            # This updates the state of the MultimodalSNIPMask modules
            self._set_active_modality(mod)
            
            # B. Forward Pass 
            # The graph records: output = weight * mask_mod
            sub_output = self.model(sub_data)
            final_outputs[mod_idx] = sub_output
            
        # C. Reset to None (safety)
        self._set_active_modality(None)
        
        return final_outputs

    def _set_active_modality(self, mod_id):
        """Helper to iterate modules and update their mask state"""
        for module in self.model.modules():
            # Check if this module has our parametrization
            if parametrize.is_parametrized(module, "weight"):
                # We need to access the specific module we registered
                # It lives in module.parametrizations.weight[0] usually
                for param_module in module.parametrizations.weight:
                    if isinstance(param_module, MultimodalSNIPMask):
                        param_module.active_mod_id = mod_id

    # --- SNIP HELPERS (Updated with BCE fix) ---
    def generate_mask_from_grad_scores(self, batch):
        scores_dict = self._calculate_scores(batch)
        threshold = self._get_threshold_from_scores(scores_dict)
        masks = {}
        for name, values in scores_dict.items():
            masks[name] = (values > threshold).float().to(self.model_device)
        return masks, None
    
    def _calculate_scores(self, batch):
        data, labels = batch
        data, labels = data.to('cpu'), labels.to('cpu')
        
        self.cpu_model.to('cpu')
        self.cpu_model.train() # Ensure train mode
        self.cpu_optimizer.zero_grad()
        
        preds = self.cpu_model(data)
        
        # Binary Cross Entropy with Logits
        loss = F.binary_cross_entropy_with_logits(preds, labels.float())
        loss.backward()
        
        scores_d = {}
        for name, module in self.cpu_model.named_modules():
            if isinstance(module, PRUNE_LAYERS) and module.weight.grad is not None:
                scores_d[name] = (module.weight.grad * module.weight.data).abs()
        return scores_d

    def _get_threshold_from_scores(self, scores_d):
        global_scores = torch.cat([torch.flatten(x) for x in scores_d.values()])
        num_params_to_keep = int(len(global_scores) * (1.0 - self.sparsity))
        if num_params_to_keep < 1: num_params_to_keep = 1 # Safety
        topk_scores, _ = torch.topk(global_scores, num_params_to_keep, sorted=True)
        return topk_scores[-1]

In [ ]:
from resnet import ResNet3D

model_channels = 64 # from config
n_classes = 1 # from config, binary

resnet_model = ResNet3D(
        in_channels=1, 
        n_classes=n_classes, 
        channels=model_channels,
    )

model = MultiMaskSNIPWrapper(resnet_model)
# model.to('cuda:0')

In [ ]:
model.register_multimodal_masks(modalities, data, labels)

# 01.23.2026 debugging gradients

In [ ]:
# get_data() gives you a test input for the model

import os
from src.db_client import ClientCreator
import numpy as np
import pandas as pd
from mindfultensors.mongoloader import MongoDataset, MongoClient
from mindfultensors.utils import unit_interval_normalize, DBBatchSampler
from catalyst.data import BatchPrefetchLoaderWrapper
from torch.utils.data import DataLoader
import torch

def make_serial(samples_for_id, kind):
    return b"".join(
        [
            sample["chunk"]
            for sample in sorted(
                (
                    sample
                    for sample in samples_for_id
                    if sample["kind"] == kind
                ),
                key=lambda x: x["chunk_id"],
            )
        ]
    )

def multimodal_collate(results, field=("input", "modality", "label")):
    """
    Use this collate function with BatchPrefetchLoaderWrapper when using MultimodalMongoDataset.
    It will stack the inputs, modality codes, and labels into tensors properly.
    """
    modality_mapping = {
        "smri": 0,
        "falff": 1,
        "dwi": 2,
    }

    # Assuming 'results' is your dictionary containing all the data
    input_tensors = [results[id_][field[0]] for id_ in results.keys()]
    modalities = [results[id_][field[1]] for id_ in results.keys()]
    label_tensors = [results[id_][field[2]] for id_ in results.keys()]
    # Stack all input tensors into a single tensor
    stacked_inputs = torch.stack(input_tensors)
    # Stack all label tensors into a single tensor
    stacked_modalities = torch.stack([torch.tensor(modality_mapping[mod]) for mod in modalities])
    stacked_labels = torch.stack(label_tensors)
    return stacked_inputs.unsqueeze(1), stacked_modalities.long(), stacked_labels.float()

def get_data(batch = list(range(1,11))):
    host = "10.245.12.58"
    host_slurm = "arctrdcn018.rs.gsu.edu"
    db_host = host_slurm if os.environ.get("SLURM_JOB_ID") else host
    crop_tensor = False
    client_creator = ClientCreator(
        db_host, crop_tensor=crop_tensor
    )

    db_name = databases = "MindfulTensors"
    db_collection = collections = "fbirn"
    meta_fields = ["gender_encoded"]
    numcubes = 1
    cubesizes = 256
    subvolume_shape = [cubesizes] * 3

    client_creator.set_database(databases)
    client_creator.set_collection(collections)
    client_creator.set_num_subcubes(numcubes)
    client_creator.set_shape(subvolume_shape)

    funcs = {
        "createclient": client_creator.create_client,
        "createVclient": client_creator.create_client,
        "mycollate": client_creator.mycollate,
        "mycollate_full": client_creator.mycollate_full,
        "mytransform": client_creator.mytransform,
    }

    db_fields = ["falff", "smri", "dwi"]

    client = MongoClient("mongodb://" + db_host + ":27017")
    db = client[db_name]
    posts_bin = db[db_collection + ".bin"]
    posts_meta = db[db_collection + ".meta"]



    fields = {"id": 1, "chunk": 1, "kind": 1, "chunk_id": 1}
    samples = list(
        posts_bin.find(
            {
                "id": {"$in": batch},
                "kind": {"$in": db_fields}, # .bin contains 3D kinds like 'smri', 'falff', 'dwi'. Scalar labels are stored in .meta
            },
            fields,
        )
    )

    results = {}

    for id in batch:
        # get ID's label and modalities
        meta_for_id = list(
            posts_meta.find(
                {
                    "id": id,
                },
                meta_fields + ["modalities"],
            )
        )

        assert len(meta_for_id) != 0, f"No meta entries found for id {id}"
        assert len(meta_for_id) < 2, f"More than one meta entry found for id {id}"

        label = meta_for_id[0][meta_fields[0]]
        modalities = meta_for_id[0]["modalities"]
        id_modalities = set(modalities).intersection(set(db_fields))

        # Get samples for this ID
        samples_for_id = [
            sample
            for sample in samples
            if sample["id"] == id
        ]

        for mod in id_modalities:
            data = make_serial(samples_for_id, mod)

            for mod in id_modalities:
                data = make_serial(samples_for_id, mod)

                result = {
                    "input": unit_interval_normalize(funcs["mytransform"](data).float()),
                    "modality": mod,
                    "label": torch.tensor(label).unsqueeze(0),
                }

                results[str(id)+'_'+mod] = result

    return multimodal_collate(results)

In [ ]:
data, modalities, labels = get_data(batch=[1,2,4,7])

data.shape, modalities.shape, labels.shape

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.parametrize as parametrize
from copy import deepcopy

# Define layers we want to prune
PRUNE_LAYERS = (nn.Linear, nn.Conv3d)

class MultimodalSNIPMask(nn.Module):
    """
    The parametrization module. It holds all masks for a specific layer
    and applies the correct one based on the active_mod_id.
    """
    def __init__(self, masks_dict):
        super().__init__()
        # We register masks as buffers so they move with .to(device) 
        # and don't get gradients.
        self.keys = sorted(list(masks_dict.keys()))
        for mod_id, mask_tensor in masks_dict.items():
            # Name format: mask_{mod_id}
            self.register_buffer(f'mask_{mod_id}', mask_tensor)
        
        # State to control which mask is active. None = Identity (no mask)
        self.active_mod_id = None

    def forward(self, weight):
        if self.active_mod_id is None:
            return weight
        
        # dynamic getattr to find the right buffer
        mask = getattr(self, f'mask_{self.active_mod_id}')
        
        # This multiplication is tracked by autograd. 
        # Backward pass will automatically mask the gradients!
        return weight * mask

class MultiMaskSNIPWrapper(nn.Module):
    def __init__(self, model, sparsity=0.9):
        super(MultiMaskSNIPWrapper, self).__init__()
        
        self.model = model
        self.sparsity = sparsity

        # Helper for SNIP calculation (frozen copy)
        self.cpu_model = deepcopy(model).to('cpu')
        self.cpu_optimizer = torch.optim.SGD(self.cpu_model.parameters(), 0.1)
        self.model_device = next(iter(model.parameters())).device
        
        # Flag to ensure we don't register twice
        self.masks_registered = False

    def register_multimodal_masks(self, modalities, input_data, labels):
        """
        1. Calculates SNIP scores.
        2. Organizes masks by layer.
        3. Registers the parametrization ONCE.
        """
        # 1. Generate Masks Dictionary: {mod_id: {layer_name: mask}}
        temp_mask_storage = {}
        unique_modalities = torch.unique(modalities).cpu().detach().tolist()
        
        for mod in unique_modalities:
            mask_idx = (modalities == mod)
            mod_data = input_data[mask_idx]
            mod_labels = labels[mask_idx]
            
            print(f"Generating masks for modality: {mod}")
            batch = (mod_data, mod_labels)
            masks_by_name, _ = self.generate_mask_from_grad_scores(batch)
            temp_mask_storage[mod] = masks_by_name

        # 2. Register Parametrizations on the actual model
        print("Registering Parametrizations...")
        for name, module in self.model.named_modules():
            if isinstance(module, PRUNE_LAYERS):
                # Collect all masks for THIS specific module
                layer_masks = {}
                has_masks = False
                for mod, mask_dict in temp_mask_storage.items():
                    if name in mask_dict:
                        layer_masks[mod] = mask_dict[name]
                        has_masks = True
                
                if has_masks:
                    # Create the parametrization module
                    snip_mask_module = MultimodalSNIPMask(layer_masks)
                    # Register it! This moves module.weight -> module.parametrizations.weight.original
                    parametrize.register_parametrization(module, "weight", snip_mask_module)
        
        self.masks_registered = True
        print("Done.")

    def forward(self, input_data, modalities):
        if not self.masks_registered:
            print("WARNING: Masks not registered. Using full weights.")
            return self.model(input_data)

        batch_size = input_data.shape[0]
        # Infer output shape or hardcode (e.g. [B, 1] for binary)
        final_outputs = torch.zeros(batch_size, 1, device=self.model_device) 
        
        unique_mods = torch.unique(modalities).cpu().tolist()

        for mod in unique_mods:
            mod_idx = (modalities == mod)
            sub_data = input_data[mod_idx]
            
            # A. Set the Active Modality
            # This updates the state of the MultimodalSNIPMask modules
            self._set_active_modality(mod)
            
            # B. Forward Pass 
            # The graph records: output = weight * mask_mod
            sub_output = self.model(sub_data)
            final_outputs[mod_idx] = sub_output
            
        # C. Reset to None (safety)
        self._set_active_modality(None)
        
        return final_outputs

    def _set_active_modality(self, mod_id):
        """Helper to iterate modules and update their mask state"""
        for module in self.model.modules():
            # Check if this module has our parametrization
            if parametrize.is_parametrized(module, "weight"):
                # We need to access the specific module we registered
                # It lives in module.parametrizations.weight[0] usually
                for param_module in module.parametrizations.weight:
                    if isinstance(param_module, MultimodalSNIPMask):
                        param_module.active_mod_id = mod_id

    # --- SNIP HELPERS (Updated with BCE fix) ---
    def generate_mask_from_grad_scores(self, batch):
        scores_dict = self._calculate_scores(batch)
        threshold = self._get_threshold_from_scores(scores_dict)
        masks = {}
        for name, values in scores_dict.items():
            masks[name] = (values > threshold).float().to(self.model_device)
        return masks, None
    
    def _calculate_scores(self, batch):
        data, labels = batch
        data, labels = data.to('cpu'), labels.to('cpu')
        
        self.cpu_model.train() # Ensure train mode
        self.cpu_optimizer.zero_grad()
        
        preds = self.cpu_model(data)
        
        # Binary Cross Entropy with Logits
        loss = F.binary_cross_entropy_with_logits(preds, labels.float())
        loss.backward()
        
        scores_d = {}
        for name, module in self.cpu_model.named_modules():
            if isinstance(module, PRUNE_LAYERS) and module.weight.grad is not None:
                scores_d[name] = (module.weight.grad * module.weight.data).abs()
        return scores_d

    def _get_threshold_from_scores(self, scores_d):
        global_scores = torch.cat([torch.flatten(x) for x in scores_d.values()])
        num_params_to_keep = int(len(global_scores) * (1.0 - self.sparsity))
        if num_params_to_keep < 1: num_params_to_keep = 1 # Safety
        topk_scores, _ = torch.topk(global_scores, num_params_to_keep, sorted=True)
        return topk_scores[-1]

In [ ]:
from resnet import ResNet3D

model_channels = 64 # from config
n_classes = 1 # from config, binary

resnet_model = ResNet3D(
        in_channels=1, 
        n_classes=n_classes, 
        channels=model_channels,
    )

model = MultiMaskSNIPWrapper(resnet_model)

In [ ]:
model.register_multimodal_masks(modalities, data, labels)
# test = model(data, modalities)

In [ ]:
def test_gradient_sparsity(wrapper_model, data, modalities, labels):
    print(f"--- Starting Gradient Sparsity Check ---")
    
    # 1. Setup Optimizer 
    optimizer = torch.optim.SGD(wrapper_model.parameters(), lr=0.1)
    optimizer.zero_grad()
    
    unique_mods = torch.unique(modalities).cpu().tolist()
    all_passed = True

    for mod in unique_mods:
        print(f"\nTesting Modality {mod}...")
        
        # A. Isolate data
        mod_idx = (modalities == mod)
        sub_data = data[mod_idx]
        sub_labels = labels[mod_idx]
        sub_mod_tensor = modalities[mod_idx]
        
        # B. Forward Pass (Wrapper handles switching internally)
        preds = wrapper_model(sub_data, sub_mod_tensor)
        
        # C. Loss & Backward
        loss = torch.nn.functional.binary_cross_entropy_with_logits(preds, sub_labels.float())
        loss.backward()
        
        # D. Verify Gradients
        total_violations = 0
        
        # We need to access the "MultimodalSNIPMask" to get the ground truth mask
        for name, module in wrapper_model.model.named_modules():
            if parametrize.is_parametrized(module, "weight"):
                
                # 1. Get the actual parameter where grads accumulate
                # With parametrize, this is usually stored here:
                param_container = module.parametrizations.weight
                original_weight = param_container.original
                grad = original_weight.grad
                
                # 2. Get the mask used for this modality
                # We dig into our custom module
                snip_module = param_container[0] # Assumes ours is the first/only parametrization
                if not isinstance(snip_module, MultimodalSNIPMask): 
                    continue
                    
                mask = getattr(snip_module, f'mask_{mod}') # Get buffer directly
                
                if grad is None:
                    print(f"  [WARN] Layer {name} has no gradient.")
                    continue
                
                # CHECK: Gradient Leakage
                leakage = grad * (1 - mask)
                min_abs_leak, max_abs_leak = torch.min(torch.abs(leakage)), torch.max(torch.abs(leakage))
                non_zero_leaks = torch.sum(torch.abs(leakage) > 1e-6).item()

                good_grads = grad * mask
                min_abs_active, max_abs_active = torch.min(torch.abs(good_grads)), torch.max(torch.abs(good_grads))
                active_count = torch.sum(torch.abs(good_grads) > 1e-6).item()
                
                if non_zero_leaks > 0:
                    print(f"  [FAIL] Layer {name}: {non_zero_leaks} leaks.")
                    all_passed = False
                    total_violations += non_zero_leaks
                else:
                   print(f"  [PASS] Layer {name}, Leak range [{min_abs_leak:.2e}, {max_abs_leak:.2e}], Active grads range [{min_abs_active:.2e}, {max_abs_active:.2e}], Active count {active_count}")

        if total_violations == 0:
            print(f"  >> Modality {mod} PASSED.")
        else:
            print(f"  >> Modality {mod} FAILED.")
        
        optimizer.zero_grad()

    return all_passed


# RUN THE TEST
is_sparse = test_gradient_sparsity(model, data, modalities, labels)
print(f"\nOverall Test Result: {'SUCCESS' if is_sparse else 'FAILURE (Leaks Detected)'}")

In [ ]:
def get_tensor_density(tensor):
    # Your function (renamed to density since we usually count non-zeros)
    nz_count = torch.count_nonzero(tensor).item()
    total_params = tensor.numel()
    return nz_count / total_params

def check_mask_sparsity(wrapper_model):
    print("\n--- Mask Sparsity Check ---")
    

    results = {}
    
    for name, module in wrapper_model.model.named_modules():
        if torch.nn.utils.parametrize.is_parametrized(module, "weight"):
            # Access our custom parametrization module
            # It's usually the first item in the container
            snip_module = module.parametrizations.weight[0] 
            
            # Iterate over the buffers we registered (mask_0, mask_1, etc.)
            for buf_name, buf_tensor in snip_module.named_buffers():
                if buf_name.startswith("mask_"):
                    mod_id = buf_name.split("_")[1]
                    density = get_tensor_density(buf_tensor)
                    sparsity = 1.0 - density
                    
                    print(f"Layer: {name} | Modality: {mod_id} | Sparsity: {sparsity:.4f}")
                    

check_mask_sparsity(model)

In [ ]:
def check_mask_intersection(wrapper_model):
    print("\n--- Mask Intersection & Diversity Check ---")
    
    for name, module in wrapper_model.model.named_modules():
        if torch.nn.utils.parametrize.is_parametrized(module, "weight"):
            snip_module = module.parametrizations.weight[0]
            
            # Extract all masks for this layer
            masks = {}
            for buf_name, buf_tensor in snip_module.named_buffers():
                if buf_name.startswith("mask_"):
                    mod_id = int(buf_name.split("_")[1]) # assuming int mod ids
                    masks[mod_id] = buf_tensor
            
            # Compare pairs
            mod_ids = sorted(list(masks.keys()))
            if len(mod_ids) < 2:
                print(f"Layer {name}: Only 1 modality, skipping intersection check.")
                continue

            print(f"Layer: {name}")
            import itertools
            for mod_a, mod_b in itertools.combinations(mod_ids, 2):
                mask_a = masks[mod_a].bool()
                mask_b = masks[mod_b].bool()
                
                # Intersection: Weights kept in BOTH
                intersection = torch.sum(mask_a & mask_b).item()
                # Union: Weights kept in EITHER
                union = torch.sum(mask_a | mask_b).item()
                
                iou = intersection / union if union > 0 else 0.0
                overlap_percent = intersection / torch.sum(mask_a).item() # How much of A is in B?
                
                density = get_tensor_density(mask_a)
                sparsity = 1.0 - density
                
                print(f"   Mod {mod_a} vs {mod_b}: IoU = {iou:.4f} | Overlap = {overlap_percent:.2%}, Sparsity: {sparsity:.3f}")

check_mask_intersection(model)

In [ ]:
def validate_active_mask_logic(wrapper_model, input_data, modalities):
    print("\n--- Validating Active Mask Logic (Live Forward Hook) ---")
    
    # 1. Pick a modality to test
    target_mod = modalities[0].item()
    print(f"Testing execution for Modality: {target_mod}")
    
    # 2. Register verification hooks
    hooks = []
    
    def verify_hook(module, args, output):
        # This runs DURING module(input)
        # At this exact moment, module.weight should be masked.
        
        # A. Get the actual weight being used
        current_weight = module.weight 
        
        # B. Manually calculate what it SHOULD be
        # We access the storage directly to get ground truth
        snip_module = module.parametrizations.weight[0]
        original_weight = module.parametrizations.weight.original
        expected_mask = getattr(snip_module, f'mask_{target_mod}')
        
        expected_weight = original_weight * expected_mask
        
        # C. Compare
        # We use allclose because floating point multiplication might have tiny diffs
        is_correct = torch.allclose(current_weight, expected_weight, atol=1e-6)
        
        status = "PASS" if is_correct else "FAIL"
        diff = (current_weight - expected_weight).abs().max().item()
        print(f"   Layer verification: {status} (Max diff: {diff:.2e})")
        
        if not is_correct:
            print("   CRITICAL FAILURE: The weight used in forward does not match (Original * Mask)!")

    # Attach hooks to a few layers
    for name, module in wrapper_model.model.named_modules():
        if torch.nn.utils.parametrize.is_parametrized(module, "weight"):
            h = module.register_forward_hook(verify_hook)
            hooks.append(h)
            
    # 3. Run a single forward pass for this modality
    # We manually simulate the wrapper loop logic for a single step
    
    # Isolate data
    mod_idx = (modalities == target_mod)
    sub_data = input_data[mod_idx]
    
    if len(sub_data) > 0:
        # ACTIVATE MASK (This is what we are testing)
        wrapper_model._set_active_modality(target_mod)
        
        # RUN FORWARD (Hooks will fire here)
        _ = wrapper_model.model(sub_data)
        
        # CLEANUP
        wrapper_model._set_active_modality(None)
    else:
        print("Skipping test: No data for this modality in batch.")

    # Remove hooks
    for h in hooks: h.remove()

validate_active_mask_logic(model, data, modalities)

# 01.20.2026 debugging masked model

In [ ]:
# get_data() gives you a test input for the model

import os
from src.db_client import ClientCreator
import numpy as np
import pandas as pd
from mindfultensors.mongoloader import MongoDataset, MongoClient
from mindfultensors.utils import unit_interval_normalize, DBBatchSampler
from catalyst.data import BatchPrefetchLoaderWrapper
from torch.utils.data import DataLoader
import torch

def make_serial(samples_for_id, kind):
    return b"".join(
        [
            sample["chunk"]
            for sample in sorted(
                (
                    sample
                    for sample in samples_for_id
                    if sample["kind"] == kind
                ),
                key=lambda x: x["chunk_id"],
            )
        ]
    )

def multimodal_collate(results, field=("input", "modality", "label")):
    """
    Use this collate function with BatchPrefetchLoaderWrapper when using MultimodalMongoDataset.
    It will stack the inputs, modality codes, and labels into tensors properly.
    """
    modality_mapping = {
        "smri": 0,
        "falff": 1,
        "dwi": 2,
    }

    # Assuming 'results' is your dictionary containing all the data
    input_tensors = [results[id_][field[0]] for id_ in results.keys()]
    modalities = [results[id_][field[1]] for id_ in results.keys()]
    label_tensors = [results[id_][field[2]] for id_ in results.keys()]
    # Stack all input tensors into a single tensor
    stacked_inputs = torch.stack(input_tensors)
    # Stack all label tensors into a single tensor
    stacked_modalities = torch.stack([torch.tensor(modality_mapping[mod]) for mod in modalities])
    stacked_labels = torch.stack(label_tensors)
    return stacked_inputs.unsqueeze(1), stacked_modalities.long(), stacked_labels.float()

def get_data(batch = list(range(1,11))):
    host = "10.245.12.58"
    host_slurm = "arctrdcn018.rs.gsu.edu"
    db_host = host_slurm if os.environ.get("SLURM_JOB_ID") else host
    crop_tensor = False
    client_creator = ClientCreator(
        db_host, crop_tensor=crop_tensor
    )

    db_name = databases = "MindfulTensors"
    db_collection = collections = "fbirn"
    meta_fields = ["gender_encoded"]
    numcubes = 1
    cubesizes = 256
    subvolume_shape = [cubesizes] * 3

    client_creator.set_database(databases)
    client_creator.set_collection(collections)
    client_creator.set_num_subcubes(numcubes)
    client_creator.set_shape(subvolume_shape)

    funcs = {
        "createclient": client_creator.create_client,
        "createVclient": client_creator.create_client,
        "mycollate": client_creator.mycollate,
        "mycollate_full": client_creator.mycollate_full,
        "mytransform": client_creator.mytransform,
    }

    db_fields = ["falff", "smri", "dwi"]

    client = MongoClient("mongodb://" + db_host + ":27017")
    db = client[db_name]
    posts_bin = db[db_collection + ".bin"]
    posts_meta = db[db_collection + ".meta"]



    fields = {"id": 1, "chunk": 1, "kind": 1, "chunk_id": 1}
    samples = list(
        posts_bin.find(
            {
                "id": {"$in": batch},
                "kind": {"$in": db_fields}, # .bin contains 3D kinds like 'smri', 'falff', 'dwi'. Scalar labels are stored in .meta
            },
            fields,
        )
    )

    results = {}

    for id in batch:
        # get ID's label and modalities
        meta_for_id = list(
            posts_meta.find(
                {
                    "id": id,
                },
                meta_fields + ["modalities"],
            )
        )

        assert len(meta_for_id) != 0, f"No meta entries found for id {id}"
        assert len(meta_for_id) < 2, f"More than one meta entry found for id {id}"

        label = meta_for_id[0][meta_fields[0]]
        modalities = meta_for_id[0]["modalities"]
        id_modalities = set(modalities).intersection(set(db_fields))

        # Get samples for this ID
        samples_for_id = [
            sample
            for sample in samples
            if sample["id"] == id
        ]

        for mod in id_modalities:
            data = make_serial(samples_for_id, mod)

            for mod in id_modalities:
                data = make_serial(samples_for_id, mod)

                result = {
                    "input": unit_interval_normalize(funcs["mytransform"](data).float()),
                    "modality": mod,
                    "label": torch.tensor(label).unsqueeze(0),
                }

                results[str(id)+'_'+mod] = result

    return multimodal_collate(results)

In [ ]:
data, modalities, labels = get_data(batch=[1,2,4,7])

data.shape, modalities.shape, labels.shape

In [ ]:
from resnet import ResNet3D, BasicBlock3D
from torch import nn
from torch.nn import functional as F
import torch

from collections import OrderedDict
from copy import deepcopy
PRUNE_LAYERS = (nn.Linear, nn.Conv3d)

class MultiMaskSNIPWrapper(nn.Module):
    def __init__(self, model, sparsity=0.9):
        super(MultiMaskSNIPWrapper, self).__init__()
        
        self.model = model
        self.sparsity = sparsity

        self.cpu_model = deepcopy(model).to('cpu')
        self.cpu_optimizer = torch.optim.SGD(self.cpu_model.parameters(), 0.1, # LR is not required for calculating scores [so far]
                                    momentum=0.9,
                                    weight_decay=1e-4)
        
        self.model_device = next(iter(model.parameters())).device

    def register_multimodal_masks(self, modalities, input_data, labels):
        """
        Generates and stores masks for each modality found in the input.
        """
        self.masks_storage = {}
        unique_modalities = torch.unique(modalities).cpu().detach().tolist()
        
        for mod in unique_modalities:
            # 1. Filter data for this modality
            mask = (modalities == mod)
            mod_data = input_data[mask]
            mod_labels = labels[mask]
            
            # 2. Generate SNIP masks
            batch = (mod_data, mod_labels)
            print(f"Generating masks for modality: {mod}")
            
            # Note: generate_mask_from_grad_scores returns (masks_by_name, masks_by_layer)
            # We only need masks_by_name for storage
            masks_by_name, _ = self.generate_mask_from_grad_scores(batch)
            
            # 3. Store in our dictionary
            self.masks_storage[mod] = masks_by_name
            
        print(f"Masks registered for modalities: {list(self.masks_storage.keys())}")

    def forward(self, input_data, modalities):
        """
        The Manual Forward Strategy:
        1. Split input by modality.
        2. Swap weights with masked weights.
        3. Run forward.
        4. Restore weights.
        """
        # Prepare an output tensor (assuming binary classification with 1 output)
        batch_size = input_data.shape[0]
        final_outputs = torch.zeros(batch_size, 1, device=self.model_device) 
        
        unique_mods = torch.unique(modalities).cpu().detach().tolist()

        for mod in unique_mods:
            # A. Identify data for this modality
            mod_idx = (modalities == mod)
            mod_data = input_data[mod_idx]

            # B. Apply the mask for this modality (Temporarily overwrite weights)
            # We can relax it in the future, it may be useful for the case 
            # when we want to train the model a little before initializing masks.
            assert mod in self.masks_storage, f"Modality {mod} not found in masks_storage"
            self._apply_mask_to_model(mod)

            # C. Run the forward pass
            # The model now thinks it has pruned weights
            mod_output = self.model(mod_data)
            
            # D. Restore original weights (CRITICAL STEP)
            self._restore_model_weights()
            
            # E. Store results in the final tensor
            final_outputs[mod_idx] = mod_output

        return final_outputs

    def _apply_mask_to_model(self, mod_id):
        masks = self.masks_storage[mod_id]
        self.original_weights_backup = {} 
        self.grad_hooks = []

        for name, module in self.model.named_modules():
            if isinstance(module, PRUNE_LAYERS):
                assert name in masks, f"Missing mask for layer {name} for modality {mod_id}"
                # 1. Back up the original weights data
                self.original_weights_backup[name] = module.weight.data.clone()
                
                # 2. Apply mask in-place to the .data attribute
                mask = masks[name]
                module.weight.data.mul_(mask)

                hook = module.weight.register_hook(lambda grad, m=mask: grad * m)
                self.grad_hooks.append(hook)

    def _restore_model_weights(self):
        for name, module in self.model.named_modules():
            if name in self.original_weights_backup:
                # Copy the backed-up values back into the weight.data
                module.weight.data.copy_(self.original_weights_backup[name])

        for hook in self.grad_hooks:
            hook.remove()
            
        self.grad_hooks = []
        self.original_weights_backup = {}

    ### mask generation section
    def generate_mask_from_grad_scores(self, batch):
        scores_dict = self._calculate_scores(batch)
        threshold = self._get_threshold_from_scores(scores_dict)
        masks = dict()
        for name, values in scores_dict.items():
            masks[name] = ((values) >= threshold).float().to(self.model_device)

        mask_key_layer = dict()
        # we have to iterate over the original model which we wish to mask, not the copied
        # cpu model, because we are using the exact layers of that model as keys to the mask
        # dictionary.
        for name, layer in self.model.named_modules(): 
            if isinstance(layer, PRUNE_LAYERS):
                mask_key_layer[layer] = masks[name]
        return masks, mask_key_layer
    
    def _calculate_scores(self, batch):
        """
        Calculate the gradient scores for later thresholding and masking.
        """
        data, labels = batch
        data, labels = data.to('cpu'), labels.to('cpu')
        # compute output
        preds = self.cpu_model(data)
        # print(f"Preds shape: {preds.shape}, \n Preds: {preds} \n labels: {labels}")
        loss = F.binary_cross_entropy_with_logits(preds, labels)
        # print(f"Loss: {loss}")
        # compute gradient and do SGD step
        self.cpu_optimizer.zero_grad()
        loss.backward()
        # Calculate the score from the weights and the gradients
        scores_d = {}
        for name, module in self.cpu_model.named_modules():
            if isinstance(module, PRUNE_LAYERS):
                # print(f"Calculating scores for layer: {name} with weight shape: {module.weight.shape}")
                # print(f"Weight min value: {module.weight.data.min()}, max value: {module.weight.data.max()}")
                # print(f"Grad min value: {module.weight.grad.min()}, max value: {module.weight.grad.max()}")
                scores_d[name] = (module.weight.grad * module.weight.data.detach()).abs()
        return scores_d

    def _get_threshold_from_scores(self, scores_d):
        """
        Calculate the threshold value from the gradient scores for masking.
        TODO: ask Riyasat if it won't lead to zeroing out some layer by accident. 
        It prolly can, but the chances are low.
        """
        global_scores = torch.cat([torch.flatten(x) for x in scores_d.values()])
        # not normalizing here, since it doesn't look like it's needed. Look here again if performance suffers.
        num_params_to_keep = int(len(global_scores) * (1.0-self.sparsity))
        topk_scores, _ = torch.topk(global_scores, num_params_to_keep, sorted=True)
        threshold = topk_scores[-1]

        # print(f"Global scores stats: Max={global_scores.max()}, Min={global_scores.min()}, Mean={global_scores.mean()}")
        # print(f"Calculated Threshold: {threshold}")
        # print(f"Targeting to keep: {num_params_to_keep} out of {len(global_scores)}")

        return threshold
nn.Module.register_full_backward_hook

In [ ]:
from resnet import ResNet3D

model_channels = 64 # from config
n_classes = 1 # from config, binary

resnet_model = ResNet3D(
        in_channels=1, 
        n_classes=n_classes, 
        channels=model_channels,
    )

model = MultiMaskSNIPWrapper(resnet_model)

In [ ]:
model.register_multimodal_masks(modalities, data, labels)
# test = model(data, modalities)

In [ ]:
import torch

def test_gradient_sparsity(wrapper_model, data, modalities, labels):
    print(f"--- Starting Gradient Sparsity Check ---")
    
    # 1. Setup Optimizer (Simple SGD just to initialize state if needed)
    optimizer = torch.optim.SGD(wrapper_model.parameters(), lr=0.1)
    optimizer.zero_grad()
    
    # 2. Iterate over modalities separately to ensure strict isolation
    unique_mods = torch.unique(modalities).cpu().tolist()
    
    # We track strict success
    all_passed = True

    for mod in unique_mods:
        print(f"\nTesting Modality {mod}...")
        
        # A. Isolate data
        mod_idx = (modalities == mod)
        sub_data = data[mod_idx]
        sub_labels = labels[mod_idx]
        sub_mod_tensor = modalities[mod_idx]
        
        # B. Forward Pass
        # We assume wrapper handles the masking internally via 'sub_mod_tensor'
        preds = wrapper_model(sub_data, sub_mod_tensor)
        
        # C. Loss & Backward
        loss = torch.nn.functional.binary_cross_entropy_with_logits(preds, sub_labels.float())
        loss.backward()
        
        # D. Verify Gradients
        total_violations = 0
        total_params_checked = 0
        
        masks = wrapper_model.masks_storage[mod]
        
        for name, module in wrapper_model.model.named_modules():
            if isinstance(module, PRUNE_LAYERS):
                
                # Get the mask and the gradient
                mask = masks[name]
                grad = module.weight.grad
                
                total_unmasked_params = torch.sum(mask == 1).item()
                total_masked_params = torch.sum(mask == 0).item()

                if grad is None:
                    print(f"  [WARN] Layer {name} has no gradient.")
                    continue
                
                # CHECK 1: Gradient Leakage
                # We expect grad to be exactly 0.0 where mask is 0.0
                # We use a tiny epsilon for float comparisons
                leakage = grad * (1 - mask)
                min_abs_leak, max_abs_leak = torch.min(torch.abs(leakage)), torch.max(torch.abs(leakage))
                non_zero_leaks = torch.sum(torch.abs(leakage) > 1e-6).item()
                
                # CHECK 2: Gradient Activity
                # We expect at least SOME gradient where mask is 1.0 (otherwise dead neuron)
                active_grads = grad * mask
                min_abs_active, max_abs_active = torch.min(torch.abs(active_grads)), torch.max(torch.abs(active_grads))
                active_count = torch.sum(torch.abs(active_grads) > 1e-6).item()
                
                total_violations += non_zero_leaks
                total_params_checked += grad.numel()
                
                if non_zero_leaks > 0:
                    print(f"  [FAIL] Layer {name}: Found {non_zero_leaks} leaks out of {total_masked_params}")
                    print(f"      Active grads: {active_count} out of {total_unmasked_params} unmasked weights.")
                    print(f"      Leakage grad abs range: [{min_abs_leak:.2e}, {max_abs_leak:.2e}]")
                    print(f"      Active grad abs range: [{min_abs_active:.2e}, {max_abs_active:.2e}]")
                    all_passed = False
                else:
                    print(f"  [PASS] Layer {name}!!!!!")

        if total_violations == 0:
            print(f"  >> Modality {mod} PASSED. No gradient leaks.")
        else:
            print(f"  >> Modality {mod} FAILED. Total leaking params: {total_violations}")
        
        # Reset grads for next modality test
        optimizer.zero_grad()

    return all_passed

# RUN THE TEST
is_sparse = test_gradient_sparsity(model, data, modalities, labels)
print(f"\nOverall Test Result: {'SUCCESS' if is_sparse else 'FAILURE (Leaks Detected)'}")

In [ ]:
# CHECK THAT GRADIENTS MATCH THE MASKS 

In [ ]:
# masks.keys()
def get_tensor_sps(tensor):
    nz_count = torch.count_nonzero(tensor).item()
    total_params = tensor.numel()

    return nz_count/total_params

for key in model.masks[list(model.masks.keys())[0]]:
    print(f"Layer: {key}, Sparsity: {100*(1 - get_tensor_sps(model.masks[list(model.masks.keys())[0]][key])):.2f} %")

In [ ]:
# backup

from resnet import ResNet3D, BasicBlock3D
from torch import nn
from torch.nn import functional as F
import torch

from collections import OrderedDict
from copy import deepcopy
PRUNE_LAYERS = (nn.Linear, nn.Conv3d)

class MultiMaskSNIPWrapper(nn.Module):
    def __init__(self, model, sparsity=0.9):
        super(MultiMaskSNIPWrapper, self).__init__()
        
        self.model = model
        self.sparsity = sparsity

        self.cpu_model = deepcopy(model).to('cpu')
        self.cpu_optimizer = torch.optim.SGD(self.cpu_model.parameters(), 0.1, # LR is not required for calculating scores [so far]
                                    momentum=0.9,
                                    weight_decay=1e-4)
        
        self.model_device = next(iter(model.parameters())).device

    def register_multimodal_masks(self, modalities, input_data, labels):
        """
        The idea if this function is to be used as the MASKs initialization step.
        Pass modalities, pass input data and labels.
        The function will compute masks based on gradients for each modality and then register them.
        """

        self.masks = {}

        # derive the masks for each modality
        # unique_modalities = torch.unique(modalities).cpu().detach().tolist()
        unique_modalities = torch.unique(modalities)
        for mod in unique_modalities:
            mod_data = input_data[modalities == mod]
            mod_labels = labels[modalities == mod]
            batch = (mod_data, mod_labels)
            print(f"Generating masks for modality: {mod} with data shape: {mod_data.shape}, labels shape: {mod_labels.shape}, unique labels: {torch.unique(mod_labels)}")
            masks, _ = self.generate_mask_from_grad_scores(batch)
            
            self.masks[mod] = masks

        # register masks in module buffers
        for name, module in self.model.named_modules():
            if isinstance(module, PRUNE_LAYERS):
                device = module.weight.get_device()
                device = 'cpu' if device == -1 else f"cuda:{device}"
                module.masks = {mod: nn.Parameter(self.masks[mod][name].requires_grad_(False).to(device)) for mod in self.masks.keys()}
                # module.masks = nn.Parameter(masks[module]).requires_grad_(False).to(module.weight.get_device())
                # hook registration is performed dynamically in the forward for each modality
                # module.register_forward_pre_hook(self._forward_pre_hook)
                

    ### mask generation section
    def generate_mask_from_grad_scores(self, batch):
        scores_dict = self._calculate_scores(batch)
        threshold = self._get_threshold_from_scores(scores_dict)
        masks = dict()
        for name, values in scores_dict.items():
            masks[name] = ((values) >= threshold).float().to(self.model_device)

        mask_key_layer = dict()
        # we have to iterate over the original model which we wish to mask, not the copied
        # cpu model, because we are using the exact layers of that model as keys to the mask
        # dictionary.
        for name, layer in self.model.named_modules(): 
            if isinstance(layer, PRUNE_LAYERS):
                mask_key_layer[layer] = masks[name]
        return masks, mask_key_layer
    
    def _calculate_scores(self, batch):
        """
        Calculate the gradient scores for later thresholding and masking.
        """
        data, labels = batch
        data, labels = data.to('cpu'), labels.to('cpu')
        # compute output
        preds = self.cpu_model(data)
        # print(f"Preds shape: {preds.shape}, \n Preds: {preds} \n labels: {labels}")
        loss = F.binary_cross_entropy_with_logits(preds, labels)
        # print(f"Loss: {loss}")
        # compute gradient and do SGD step
        self.cpu_optimizer.zero_grad()
        loss.backward()
        # Calculate the score from the weights and the gradients
        scores_d = {}
        for name, module in self.cpu_model.named_modules():
            if isinstance(module, PRUNE_LAYERS):
                # print(f"Calculating scores for layer: {name} with weight shape: {module.weight.shape}")
                # print(f"Weight min value: {module.weight.data.min()}, max value: {module.weight.data.max()}")
                # print(f"Grad min value: {module.weight.grad.min()}, max value: {module.weight.grad.max()}")
                scores_d[name] = (module.weight.grad * module.weight.data.detach()).abs()
        return scores_d

    def _get_threshold_from_scores(self, scores_d):
        """
        Calculate the threshold value from the gradient scores for masking.
        TODO: ask Riyasat if it won't lead to zeroing out some layer by accident. 
        It prolly can, but the chances are low.
        """
        global_scores = torch.cat([torch.flatten(x) for x in scores_d.values()])
        # not normalizing here, since it doesn't look like it's needed. Look here again if performance suffers.
        num_params_to_keep = int(len(global_scores) * (1.0-self.sparsity))
        topk_scores, _ = torch.topk(global_scores, num_params_to_keep, sorted=True)
        threshold = topk_scores[-1]

        # print(f"Global scores stats: Max={global_scores.max()}, Min={global_scores.min()}, Mean={global_scores.mean()}")
        # print(f"Calculated Threshold: {threshold}")
        # print(f"Targeting to keep: {num_params_to_keep} out of {len(global_scores)}")

        return threshold

    ### hooks

    def _unregister_mask(self):
        for module in self.model.modules():
            module._backward_hooks = OrderedDict()
            module._forward_pre_hooks = OrderedDict()


    @staticmethod
    def _forward_pre_hook(module, x):
        module.mask.requires_grad_(False)
        mask = module.mask
        module.weight.data.mul_(mask.to(module.weight.get_device())) # This might be a place of performance loss because to -> .to()


    # def get_model_sps(self):
    #     nonzero = total = 0
    #     for name, param in self.model.named_parameters():
    #         if 'mask' not in name:
    #             tensor = param.detach().clone()
    #             # nz_count.append(torch.count_nonzero(tensor))
    #             nz_count = torch.count_nonzero(tensor).item()
    #             total_params = tensor.numel()
    #             nonzero += nz_count
    #             total += total_params
        
    #     tensor = None
    #     abs_sps = 100 * (total-nonzero) / total
    #     return abs_sps
    
    # def print_model_sps(self):
    #     print(f"Model Sparsity: {self.get_model_sps():.2f} %")

    # def forward(self, x):
    #     mask = self.masks.iloc[self.current_mask_index].values
    #     mask_tensor = {name: torch.tensor(mask_value, dtype=torch.float32) 
    #                    for name, mask_value in zip(self.model.state_dict().keys(), mask)}
        
    #     for name, param in self.model.named_parameters():
    #         if name in mask_tensor:
    #             param.data *= mask_tensor[name]
        
    #     return self.model(x)
    

# 01.12.2026 updating for mutlimodal training

In [ ]:
import os
from src.db_client import ClientCreator

host = "10.245.12.58"
host_slurm = "arctrdcn018.rs.gsu.edu"
db_host = host_slurm if os.environ.get("SLURM_JOB_ID") else host
crop_tensor = False
client_creator = ClientCreator(
    db_host, crop_tensor=crop_tensor
)

db_name = databases = "MindfulTensors"
db_collection = collections = "fbirn"
meta_fields = ["gender_encoded"]
index_id = 'id'
numcubes = 1
cubesizes = 256
subvolume_shape = [cubesizes] * 3
shape = subvolume_shape[0]
numvolumes = num_volumes = 4
prefetches = 16

client_creator.set_database(databases)
client_creator.set_collection(collections)
client_creator.set_num_subcubes(numcubes)
client_creator.set_shape(subvolume_shape)

funcs = {
    "createclient": client_creator.create_client,
    "createVclient": client_creator.create_client,
    "mycollate": client_creator.mycollate,
    "mycollate_full": client_creator.mycollate_full,
    "mytransform": client_creator.mytransform,
}


In [ ]:
from mindfultensors.mongoloader import MongoDataset, MongoClient
from mindfultensors.utils import unit_interval_normalize, DBBatchSampler
from catalyst.data import BatchPrefetchLoaderWrapper
from torch.utils.data import DataLoader

db_fields = ["falff"]
db_fields = ["smri"]
db_fields = ["dwi"]
db_fields = ["falff", "smri", "dwi"]

client = MongoClient("mongodb://" + db_host + ":27017")
db = client[db_name]
posts_bin = db[db_collection + ".bin"]
posts_meta = db[db_collection + ".meta"]

all_ids = posts_meta.distinct( # pull all unique IDs (subjects) with at least one modality in db_fields
    "id",
    {'modalities': {"$in": db_fields}}
)
all_ids = sorted(all_ids)
# print(all_ids)

labels = []
for id in all_ids:
    label = posts_meta.find_one({"id": id}, meta_fields)[meta_fields[0]] # get label for the id
    labels.append(label)
labels = np.array(labels)
labels.shape, len(all_ids)

In [ ]:
# checking the samples, need to figure out how to stitch modalities together correctly

batch = [1, 2, 3, 4]

metas = list(posts_meta.find({
    "id": {"$in": batch}
}))

bins = list(posts_bin.find(
    {
    "id": {"$in": batch}
    },
))

# len(metas), len(bins), metas[1]
bins[0].keys(), metas[0].keys(), metas[0]["modalities"]

In [ ]:
from mindfultensors.mongoloader import MongoDataset
import torch
from mindfultensors.utils import unit_interval_normalize
from torch.utils.data import TensorDataset

class MultimodalMongoDataset(MongoDataset):
    def __init__(
        self,
        indices,
        transform,
        collection,
        sample,
        meta_sample,
        normalize=unit_interval_normalize,
        id="id",
    ):
        super(MultimodalMongoDataset, self).__init__(indices, transform, collection, sample, normalize, id)
        self.meta_sample = meta_sample

    def __getitem__(self, batch):
        # Fetch all samples for ids in the batch and where 'kind' is either
        # data or label as specified by the sample parameter
        # TODO: make it respect the batch size; right now it returns a bigger batch with multiple modalities per id
        
        samples = list(
            self.collection["bin"].find(
                {
                    self.id: {"$in": [self.indices[_] for _ in batch]},
                    "kind": {"$in": self.sample}, # .bin contains 3D kinds like 'smri', 'falff', 'dwi'. Scalar labels are stored in .meta
                },
                self.fields,
            )
        )

        results = {}
        for id in batch:
            # get ID's label and modalities
            meta_for_id = list(
                self.collection["meta"].find(
                    {
                        self.id: self.indices[id],
                    },
                    self.meta_sample + ["modalities"],
                )
            )

            assert len(meta_for_id) != 0, f"No meta entries found for id {id}"
            assert len(meta_for_id) < 2, f"More than one meta entry found for id {id}"
            
            label = meta_for_id[0][self.meta_sample[0]]
            modalities = meta_for_id[0]["modalities"]
            id_modalities = set(modalities).intersection(set(self.sample))
                

            # Get samples for this ID
            samples_for_id = [
                sample
                for sample in samples
                if sample[self.id] == self.indices[id]
            ]

            for mod in id_modalities:
                data = self.make_serial(samples_for_id, mod)

                result = {
                    "input": self.normalize(self.transform(data).float()),
                    "modality": mod,
                    "label": torch.tensor(label).unsqueeze(0),
                }


                # Add to results
                results[str(id)+'_'+mod] = result

        return results

In [ ]:
# info on collate mystery 

crop_tensor = False

cubesizes = 256
subvolume_shape = [cubesizes] * 3
shape = subvolume_shape[0]

def mycollate_full(self, x):
    return crop_tensor(*mcollate(x)) if self.crop_tensor else mcollate(x)

def mcollate(results, field=("input", "label")):
    results = results[0]
    # Assuming 'results' is your dictionary containing all the data
    input_tensors = [results[id_][field[0]] for id_ in results.keys()]
    label_tensors = [results[id_][field[1]] for id_ in results.keys()]
    # Stack all input tensors into a single tensor
    stacked_inputs = torch.stack(input_tensors)
    # Stack all label tensors into a single tensor
    stacked_labels = torch.stack(label_tensors)
    return stacked_inputs.unsqueeze(1), stacked_labels.long()

funcs = {
    "createclient": client_creator.create_client,
    "createVclient": client_creator.create_client,
    "mycollate": client_creator.mycollate,
    "mycollate_full": client_creator.mycollate_full,
    "mytransform": client_creator.mytransform,
}

collate = (
    funcs["mycollate_full"]
    if shape == 256
    else funcs["mycollate"]
)

In [ ]:
from src.customMongoDataset import CustomMongoDataset

# train_dataset = CustomMongoDataset(
train_dataset = MultimodalMongoDataset(
    all_ids, 
    funcs["mytransform"],
    None,
    db_fields,
    meta_fields,
    normalize=unit_interval_normalize,
    id=index_id,
)

def multimodal_collate(results, field=("input", "modality", "label")):
    modality_mapping = {
        "smri": 0,
        "falff": 1,
        "dwi": 2,
    }
    results = results[0]
    # Assuming 'results' is your dictionary containing all the data
    input_tensors = [results[id_][field[0]] for id_ in results.keys()]
    modalities = [results[id_][field[1]] for id_ in results.keys()]
    label_tensors = [results[id_][field[2]] for id_ in results.keys()]
    # Stack all input tensors into a single tensor
    stacked_inputs = torch.stack(input_tensors)
    # Stack all label tensors into a single tensor
    stacked_modalities = torch.stack([torch.tensor(modality_mapping[mod]) for mod in modalities])
    stacked_labels = torch.stack(label_tensors)
    return stacked_inputs.unsqueeze(1), stacked_modalities.long(), stacked_labels.long()

collate = multimodal_collate

train_sampler = DBBatchSampler(train_dataset, batch_size=num_volumes)
train_dataloader = BatchPrefetchLoaderWrapper(
    DataLoader(
        train_dataset,
        sampler=train_sampler,
        collate_fn=collate,
        pin_memory=True,
        worker_init_fn=funcs["createclient"],
        persistent_workers=True,
        prefetch_factor=1,
        num_workers=1,  # prefetches,
        # prefetch_factor=None,
        # num_workers=1,  # prefetches,
    ),
    num_prefetches=prefetches,
)

In [ ]:
for batch in train_dataloader:
    print(len(batch))
    print(batch[0].shape)
    print(batch[1])
    break